# 03 · Build a portable geospatial layer

**Mission:** create and validate GeoJSON with units, provenance, and freshness, then import it into both viewers. Allow 30 minutes. Prerequisite: Lab 00.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then Run → Run All Cells. First use downloads Matplotlib. All training data are **SYNTHETIC** unless explicitly labeled LIVE. Downloads appear below and in `exports/`; the browser filesystem is separate from your computer. Each lab runs independently.

In [1]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install("matplotlib")
import json, math
from pathlib import Path
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from worldview_lab import (load, export, map_layer, chart_style, haversine_km,
    wilson_interval, shortest_times, freshness, normalize_quakes,
    validate_geojson, brief_request, validate_brief)
chart_style()

In [2]:
sites=load('facilities.geojson')
for feature in sites['features']:
    p=feature['properties']
    p['freshness']=freshness(p['observed_at'],'2026-09-14T12:00:00Z')
assert validate_geojson(sites)
map_layer(sites,value='capacity',title='SYNTHETIC facilities — capacity in exercise service slots')
export('my-facilities.geojson',sites)

<iframe title="SYNTHETIC facilities — capacity in exercise service slots" referrerpolicy="strict-origin-when-cross-origin" style="width:100%;height:520px;border:0" srcdoc="<!doctype html><html lang="en"><meta charset="utf-8">
 <meta name="referrer" content="strict-origin-when-cross-origin">
 <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
 <style>body{margin:0;font:14px system-ui}#map{height:390px;background:#eef3f5}header{padding:10px;background:#102536;color:#fff}.controls{padding:8px;background:#f2f6f8}#map-status{display:block;margin-top:4px}</style>
 <header>SYNTHETIC facilities — capacity in exercise service slots | Select a feature to inspect its attributes and units</header>
 <div class="controls"><label><input id="basemap" type="checkbox"> OpenStreetMap background</label>
 <span id="map-status" role="status" aria-live="polite">Data layer ready. Background map off.</span></div>
 <div id="map" aria-label="Interactive geospatial data layer"></div>
 <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script><script>
 const data={"type": "FeatureCollection", "features": [{"type": "Feature", "id": "F01", "geometry": {"type": "Point", "coordinates": [-77.23, 38.71]}, "properties": {"id": "F01", "name": "Harbor exercise clinic", "capacity": 70, "open": true, "status": "SIMULATED", "source": "SYNTHETIC exercise v1", "observed_at": "2026-09-14T11:30:00Z", "freshness": "CURRENT"}}, {"type": "Feature", "id": "F02", "geometry": {"type": "Point", "coordinates": [-77.1, 38.83]}, "properties": {"id": "F02", "name": "Central exercise hospital", "capacity": 110, "open": true, "status": "SIMULATED", "source": "SYNTHETIC exercise v1", "observed_at": "2026-09-14T11:30:00Z", "freshness": "CURRENT"}}, {"type": "Feature", "id": "F03", "geometry": {"type": "Point", "coordinates": [-76.84, 38.95]}, "properties": {"id": "F03", "name": "Ridge exercise clinic", "capacity": 55, "open": true, "status": "SIMULATED", "source": "SYNTHETIC exercise v1", "observed_at": "2026-09-14T11:30:00Z", "freshness": "CURRENT"}}, {"type": "Feature", "id": "F04", "geometry": {"type": "Point", "coordinates": [-76.91, 38.71]}, "properties": {"id": "F04", "name": "South exercise shelter", "capacity": 90, "open": false, "status": "SIMULATED", "source": "SYNTHETIC exercise v1", "observed_at": "2026-09-14T11:30:00Z", "freshness": "CURRENT"}}]},field="capacity",m=L.map('map').setView([38.85,-77.04],10);
 const layer=L.geoJSON(data,{style:f=>({color:'#15364a',weight:2,fillOpacity:.65,fillColor:f.properties[field]>=400?'#ef806f':f.properties[field]>=200?'#f4bc65':'#54c9c1'}),pointToLayer:(f,ll)=>L.circleMarker(ll,{radius:8,color:'#157b85'})});
 layer.eachLayer(l=>{const pre=document.createElement('pre');pre.textContent=JSON.stringify(l.feature.properties,null,2);l.bindPopup(pre)});layer.addTo(m);if(data.features.length)m.fitBounds(layer.getBounds());
 const toggle=document.getElementById('basemap'),status=document.getElementById('map-status');
 const tiles=L.tileLayer('https://tile.openstreetmap.org/{z}/{x}/{y}.png',{
 maxZoom:18,attribution:'© <a href="https://www.openstreetmap.org/copyright">OpenStreetMap contributors</a>'});
 tiles.on('tileerror',()=>{
 if(!m.hasLayer(tiles))return;
 m.removeLayer(tiles);toggle.checked=false;
 status.textContent='Background map unavailable. Your data remain visible. Select OpenStreetMap background to retry.';
 });
 tiles.on('load',()=>{if(m.hasLayer(tiles))status.textContent='Data layer and OpenStreetMap background ready.'});
 toggle.addEventListener('change',()=>{
 if(toggle.checked){status.textContent='Loading OpenStreetMap background…';tiles.addTo(m)}
 else{m.removeLayer(tiles);status.textContent='Data layer ready. Background map off.'}
 });
 // file:// and opaque embedding contexts cannot identify a referring website.
 if(/^https?:/.test(document.baseURI)){
 toggle.checked=true;if(toggle.checked)toggle.dispatchEvent(new Event('change'));
 }else{
 toggle.disabled=true;
 status.textContent='Da

WindowsPath('exports/my-facilities.geojson')

## Background tiles and saved browser files

The **OpenStreetMap background** checkbox controls the street map beneath your facilities. If tiles are unavailable, the map keeps your four synthetic points and their popups visible and explains the failure. Check the box again to retry, or call `map_layer(sites, value="capacity", basemap=False)` to work with just the data layer. This still needs the Leaflet library; it does not download offline street maps.

If an older copy shows **Access blocked** tiles, open this lab in a private browser window to load the updated helper and notebook without deleting your saved work. JupyterLite retains browser copies, so refreshing alone may keep the old `worldview_lab.py`. If you replace that file in an existing session, restart the Python kernel before running all cells.

The renderer preserves the browser's real referring origin, as required by the [OpenStreetMap tile usage policy](https://operations.osmfoundation.org/policies/tiles/). It removes a failed background layer instead of repeatedly retrying. Third-party tile services can still be unavailable. The one-time Matplotlib font-cache message is normal initialization, not a map error.

## Coordinate order and provenance

GeoJSON positions use **longitude, latitude**; Leaflet’s direct `LatLng` API uses **latitude, longitude**. `L.geoJSON` performs the conversion. Values in range can still be swapped, so also check a known location and map extent. These points are invented facilities, not a directory.

Download the layer, open **2D EOC → Import GeoJSON** or **3D scenes → Import GeoJSON**, and select it. The import is local to that browser tab. Only Point and Polygon geometries are accepted by this teaching importer; a rejected file reports a reason.

In [3]:
import copy
bad=copy.deepcopy(sites)
bad['features'][0]['geometry']['coordinates']=[400,95]
try: validate_geojson(bad)
except ValueError as error: print('Expected rejection:',error)
print('A stale observation:',freshness('2026-09-14T08:00:00Z','2026-09-14T12:00:00Z'))

Expected rejection: Coordinates out of range
A stale observation: STALE


## Lab challenge

Add a fifth fictional facility with a source, UTC observation time, and capacity. Change the scenario clock by two hours and compare freshness. **Checkpoint:** all five points appear in both viewers; bad coordinates are rejected; no source labels disappear.

**Astra prompt:** “Review this GeoJSON schema and validation code. Propose edge cases for coordinate order, null geometry, missing source, and future timestamps. Explain each failure before proposing a fix.”

## Sources and attribution

[RFC 7946](https://www.rfc-editor.org/rfc/rfc7946) defines GeoJSON. [Leaflet reference](https://leafletjs.com/reference.html) documents GeoJSON ingestion.

World View inspiration: [Kevin (Khoa) To, MIT](https://github.com/kevtoe/worldview). Original lab code, figures, and synthetic fixtures: JupyterLite WorldView contributors, MIT. See the [book references](https://jltobias.github.io/JupyterLite-WorldView/book/references.html) and repository notices for dependency and service terms.